# 📘 GraphRAG — Live Demo Session
### RAG vs GraphRAG: From Theory to Proof

> **Simform Internal Session** | Run each cell top to bottom | All outputs are live

---

**What you'll see in this notebook:**
1. Why RAG exists and where it breaks
2. How GraphRAG fixes those gaps
3. Live demo on a real AI research paper
4. Side-by-side query comparison with real outputs
5. Clear pros, cons, and use-case decision guide

**Tech Stack:**
- 📄 Paper: [LLM Agents - arXiv 2304.03442](https://arxiv.org/pdf/2304.03442.pdf)
- 🗄️ Vector DB: ChromaDB (for RAG)
- 🕸️ Graph DB: Neo4j Aura (for GraphRAG)
- 🤖 LLM: GPT-4o-mini via OpenAI
- 🔗 Orchestration: LangChain

---
## 🔹 PART 1: THEORY
### Section 1 & 2 — RAG Basics & GraphRAG Basics

---

### What is RAG?

**RAG = Retrieval-Augmented Generation**

The idea is simple: instead of relying on what the LLM memorized during training, we *retrieve* relevant text from your documents at query time and hand it to the LLM as context.

```
User Query → Embed → Similarity Search → Top-K Chunks → LLM → Answer
```

**Core mechanism — vector similarity:**
- Every document chunk is converted to a vector (embedding)
- The query is also embedded
- Cosine similarity finds the most *similar* chunks
- Those chunks become the context for the LLM

✅ Works well for: *"What is memory in LLM agents?"* (answer lives in one paragraph)

---

### What is GraphRAG?

**GraphRAG = RAG + Knowledge Graph**

Instead of chunking and embedding text, we first **extract entities and relationships** from documents and build a **knowledge graph**. Queries then traverse that graph to find connected information.

```
Documents → Extract Entities + Relationships → Neo4j Graph
     
User Query → Extract Entities → Cypher Traversal → Graph Paths → LLM → Answer
```

**Core mechanism — graph traversal:**
- Entities (agent, memory, tools) = nodes in the graph
- Relationships (uses, depends on) = edges between nodes
- A query finds nodes matching the question and follows edges
- The *path* of connections becomes the context for the LLM

✅ Works well for: *"How does memory help planning?"* (requires connecting two separate concepts)

---
### Section 3 — Core Differences

| Dimension | RAG | GraphRAG |
|-----------|-----|----------|
| Retrieval method | Vector similarity | Graph traversal |
| Knowledge format | Flat text chunks | Structured nodes + edges |
| Reasoning | Weak — single-hop | Strong — multi-hop |
| Explainability | Low | High — shows the path |
| Setup complexity | Simple | Complex |
| Speed (query) | Fast | Slower |

---

### Section 4 — Why GraphRAG Exists

**Problems RAG cannot solve:**

1. **Cross-document context gap** — RAG retrieves chunks but cannot connect facts from different sections/documents
2. **Concept linking failure** — "Memory helps planning" requires relating two concepts that may never co-appear in one chunk
3. **Multi-hop query failure** — "A uses B which improves C" requires chained reasoning; RAG returns fragmented pieces
4. **Scattered information** — Top-k similar chunks can be *redundant* and completely miss the connecting logic

**How GraphRAG solves these:**

1. Single unified graph across ALL documents — everything is connected
2. Relationships are explicit edges — no inference needed
3. Graph traversal naturally follows A → B → C chains
4. Compact structured paths instead of raw repetitive text

---

### Section 5 — Core Concepts

```
Entity       = Any object of interest: "agent", "memory", "tools", "planning"
Node         = Entity stored in the graph (1 entity = 1 node)
Relationship = A named connection: "uses", "depends on", "improves"
Edge         = Directed relationship stored as: source ──[relation]──► target
Graph        = Total collection of nodes + edges = your knowledge base
Multi-hop    = Query that traverses multiple edges:
               agent ──uses──► tools ──improve──► reasoning
```

**From our demo paper — actual triplets extracted:**

| Subject | Predicate | Object |
|---------|-----------|--------|
| agent | uses | tools |
| agent | relies on | memory |
| planning | improves | reasoning |
| memory | stores | past actions |
| tools | extend | agent capabilities |
| planning | decomposes | tasks |

---

### Section 6 — End-to-End Pipeline

**RAG Pipeline:**
```
Load PDF → Split chunks → Embed → ChromaDB → Query: embed → similarity → top-3 → GPT-4o-mini
```

**GraphRAG Pipeline:**
```
Load PDF → Split chunks → LLM extracts (subject, predicate, object) triplets 
         → Store in Neo4j
         → Query: extract entities → Cypher traverse → graph paths → GPT-4o-mini
```

---
## 🔹 PART 2: SETUP

Run these cells once at the start. They load environment credentials, verify the Neo4j connection, and download the demo paper. Everything reads from the `.env` file — no hardcoded keys anywhere.

In [2]:
# ── Cell 1: Imports & Environment Setup ──
import os
import re
import json
import requests

from dotenv import load_dotenv
load_dotenv()

# ── Credentials from .env ──
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ── Sanity check (masks secrets) ──
print("✅ Environment loaded")
print(f"   NEO4J_URI      : {NEO4J_URI}")
print(f"   NEO4J_USERNAME : {NEO4J_USERNAME}")
print(f"   NEO4J_DATABASE : {NEO4J_DATABASE}")
print(f"   OPENAI_API_KEY : {OPENAI_API_KEY[:20]}...{OPENAI_API_KEY[-4:]}")

✅ Environment loaded
   NEO4J_URI      : neo4j+s://3a18d51c.databases.neo4j.io
   NEO4J_USERNAME : 3a18d51c
   NEO4J_DATABASE : 3a18d51c
   OPENAI_API_KEY : sk-proj-8UkVueImU_e-...Mr4A


In [3]:
# ── Cell 2: Test Neo4j Connection ──
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("RETURN 'Neo4j connected!' AS message")
        record = result.single()
        print(f"✅ {record['message']}")
        print(f"   Connected to: {NEO4J_URI}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Neo4j connected!
   Connected to: neo4j+s://3a18d51c.databases.neo4j.io


In [5]:
# ── Cell 3: Download & Load the Demo Paper ──
# Paper: "A Survey on Large Language Model based Autonomous Agents" (arXiv 2304.03442)

PDF_PATH = "llm_agents.pdf"
PDF_URL  = "https://arxiv.org/pdf/2304.03442.pdf"

if not os.path.exists(PDF_PATH):
    print(f"⬇️  Downloading paper from arXiv...")
    response = requests.get(PDF_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    response.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(response.content)
    print(f"✅ Downloaded → {PDF_PATH}  ({len(response.content)//1024} KB)")
else:
    print(f"✅ Paper already exists → {PDF_PATH}")

# Load with LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader   = PyPDFLoader(PDF_PATH)
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
chunks   = splitter.split_documents(raw_docs)

print(f"\n📄 Paper loaded:")
print(f"   Pages  : {len(raw_docs)}")
print(f"   Chunks : {len(chunks)}")
print(f"\n📝 Sample chunk (first 300 chars):")
print(f"   {chunks[2].page_content[:300]}...")

✅ Paper already exists → llm_agents.pdf

📄 Paper loaded:
   Pages  : 22
   Chunks : 243

📝 Sample chunk (first 300 chars):
   spaces for interpersonal communication to prototyping tools. In
this paper, we introduce generative agents: computational software
agents that simulate believable human behavior. Generative agents
wake up, cook breakfast, and head to work; artists paint, while
Permission to make digital or hard copi...


---
## 🔹 PART 3: RAG PIPELINE

### How RAG Works (live walkthrough)

```
  chunks ──► OpenAI Embeddings ──► ChromaDB (vector index)
                                          │
  Query ──► embed ──► cosine similarity ──► top-3 matching chunks ──► GPT-4o-mini ──► Answer
```

**Key parameters:**
- `chunk_size=600` — how many characters per chunk (trade-off: too small = missing context, too large = noise)
- `chunk_overlap=60` — overlap between consecutive chunks to avoid cutting sentences
- `k=3` — how many chunks to retrieve per query

**What ChromaDB stores:**
- The raw text of each chunk
- The embedding vector for that chunk
- Metadata (page number, source file)

In [6]:
# ── Cell 4: Build the RAG Pipeline ──
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Embeddings model
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, model="text-embedding-ada-002")

# 2. Build ChromaDB vector store from chunks
print("⏳ Building ChromaDB vector store (this embeds all chunks)...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_graphrag_demo"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. LLM
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

# 4. RAG prompt — instructs LLM to use ONLY the retrieved context
rag_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Answer the question using ONLY the provided context.
If the context is insufficient, explicitly say what's missing.

Context:
{context}

Question: {question}

Answer (be specific, cite the context):
""")

def format_docs(docs):
    return "\n\n---\n\n".join([f"[Chunk {i+1}]:\n{d.page_content}" for i, d in enumerate(docs)])

# 5. LCEL RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

def rag_query(question: str) -> dict:
    """Run a RAG query and return answer + source chunks."""
    retrieved = retriever.invoke(question)
    answer    = rag_chain.invoke(question)
    return {
        "answer":  answer,
        "chunks":  [d.page_content[:200] + "..." for d in retrieved]
    }

print("✅ RAG pipeline ready")
print(f"   Vector store chunks indexed: {vectorstore._collection.count()}")
print(f"   Retriever k=3 (returns top 3 chunks per query)")

⏳ Building ChromaDB vector store (this embeds all chunks)...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ RAG pipeline ready
   Vector store chunks indexed: 243
   Retriever k=3 (returns top 3 chunks per query)


---
### 🟢 Simple Queries — RAG handles these well

These are direct, factual questions where the answer lives in a single paragraph in the paper. RAG is well-suited here — the embedding finds the right chunk and the LLM has exactly what it needs.

**Queries we'll run:**
1. `What is an autonomous agent?`
2. `What are the components of LLM agents?`
3. `What is memory in agents?`

In [7]:
# ── Cell 5: RAG — Simple Queries ──

simple_queries = [
    "What is an autonomous agent?",
    "What are the components of LLM agents?",
    "What is memory in agents?"
]

print("=" * 65)
print("🔵  RAG — SIMPLE QUERIES")
print("=" * 65)

for q in simple_queries:
    result = rag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"\n💬 RAG Answer:\n{result['answer']}")
    print(f"\n📎 Retrieved chunks used:")
    for i, chunk in enumerate(result["chunks"], 1):
        print(f"   [{i}] {chunk}")
    print("\n" + "-" * 65)

print("\n✅ Observation: RAG answers these well — answers live in single chunks")

🔵  RAG — SIMPLE QUERIES


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



❓ Query: What is an autonomous agent?

💬 RAG Answer:
An autonomous agent is a computational software agent that can simulate believable human behavior, as described in Chunk 3. These agents can perform actions such as waking up, cooking breakfast, and heading to work, indicating their ability to operate independently and interact with their environment.

📎 Retrieved chunks used:
   [1] for language- and agent-based interaction in human-computer in-
teraction. Formative work such as SHRDLU [103] and ELIZA [102]
demonstrated the opportunities and the risks associated with nat-
ural la...
   [2] will treat the user-controlled agent no differently than they treat
each other. They recognize its presence, initiate interactions, and
remember its behavior before forming opinions about it.
Users an...
   [3] spaces for interpersonal communication to prototyping tools. In
this paper, we introduce generative agents: computational software
agents that simulate believable human behavior. Generativ

---
### 🔴 Multi-hop Queries — Where RAG Breaks

These questions require *connecting* two or more concepts that appear in different parts of the paper. RAG retrieves the most similar text, but that's not enough — the connection itself isn't written in any single chunk.

Look at the output and note:
- Each retrieved chunk addresses only one concept
- The "because" / "therefore" reasoning is absent
- The LLM has to guess the connection — it was never given it

**Queries:**
1. `How does memory help planning?`
2. `How are tools, memory, and planning connected?`
3. `Why do agents need both tools and memory?`
4. `How do tools improve reasoning via planning?`

In [8]:
# ── Cell 6: RAG — Multi-hop Queries (shows the limitation) ──

multihop_queries = [
    "How does memory help planning?",
    "How are tools, memory, and planning connected?",
    "Why do agents need both tools and memory?",
    "How do tools improve reasoning via planning?"
]

print("=" * 65)
print("🔵  RAG — MULTI-HOP QUERIES  (watch for limitations)")
print("=" * 65)

for q in multihop_queries:
    result = rag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"\n💬 RAG Answer:\n{result['answer']}")
    print(f"\n📎 Retrieved chunks used:")
    for i, chunk in enumerate(result["chunks"], 1):
        print(f"   [{i}] {chunk}")
    print("\n" + "-" * 65)

print("""
⚠️  RAG LIMITATION OBSERVED:
   ✗ Answers address concepts individually, not their CONNECTION
   ✗ Retrieved chunks are often about only ONE of the two concepts
   ✗ The "how" and "why" of the relationship is missing
   ✗ LLM has to guess the connection — it's not in the context
""")

🔵  RAG — MULTI-HOP QUERIES  (watch for limitations)

❓ Query: How does memory help planning?

💬 RAG Answer:
Memory helps planning by storing reflections and plans, which are then included in the retrieval process. This allows the agent to consider observations, reflections, and plans together, enabling it to draw conclusions about itself and others to guide its behavior more effectively. Specifically, as mentioned in Chunk 1, reflections and plans are fed back into the memory stream to influence the agent’s future behavior, thereby enhancing the planning process.

📎 Retrieved chunks used:
   [1] The second is reflection, which synthesizes memories into higher-
level inferences over time, enabling the agent to draw conclusions
about itself and others to better guide its behavior. The third is
...
   [2] includes a location, a starting time, and a duration. For instance,
Klaus Mueller, who is dedicated in his research and has an im-
pending deadline,5 may choose to spend his day working 

---
## 🔹 PART 4: GRAPHRAG PIPELINE

### How GraphRAG Works (live walkthrough)

**Step 1 — Build the Knowledge Graph (one-time preprocessing):**
```
For each chunk:
   LLM reads the chunk and extracts triplets:
   [{"subject": "agent", "predicate": "uses", "object": "tools"}, ...]
   
   → MERGE each entity as a Node in Neo4j
   → MERGE each relationship as an Edge in Neo4j
```

**Step 2 — Query the Knowledge Graph:**
```
User question → LLM extracts key entities → ["memory", "planning"]
→ Cypher query finds all paths from those entities (1-2 hops)
→ Returns: memory → stores → past actions, planning → uses → past actions
→ Those structured paths = the LLM's context
→ LLM answers using the relational context
```

**Neo4j Cypher Query used:**
```cypher
MATCH (e:Entity)-[r:RELATES]->(t:Entity)
WHERE ANY(entity IN $entities 
      WHERE toLower(e.name) CONTAINS entity 
         OR toLower(t.name) CONTAINS entity)
RETURN e.name AS source, r.type AS relation, t.name AS target
LIMIT 25
```

> **Why MERGE instead of CREATE?**  
> `MERGE` is idempotent — if the same entity/relationship appears in multiple chunks, it's deduplicated automatically. This is how the graph stays clean.

In [9]:
# ── Cell 7: Build the Knowledge Graph in Neo4j ──
# This is the key step that makes GraphRAG different from RAG.
# We use an LLM to read each chunk and extract (subject, predicate, object) triplets.

extract_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

EXTRACTION_PROMPT = """You are a knowledge graph builder.
Extract the most important knowledge triplets from this text about LLM agents.
Focus on relationships between these concepts: agent, memory, tools, planning, reasoning, 
task, environment, execution, perception, learning, action.

Return ONLY a valid JSON array. Each item: {{"subject": "...", "predicate": "...", "object": "..."}}
- Use lowercase for all values
- Keep entities short (1-3 words)
- Use action verbs as predicates (uses, relies on, improves, stores, enables, requires)
- Extract 3-7 triplets maximum

Text:
{text}

JSON array:"""


def extract_triplets(text: str) -> list[dict]:
    """Use LLM to extract (subject, predicate, object) triplets from text."""
    prompt  = EXTRACTION_PROMPT.format(text=text[:1200])
    response = extract_llm.invoke(prompt)
    raw      = response.content.strip()
    match    = re.search(r'\[.*\]', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            return []
    return []


def store_triplet(tx, subject: str, predicate: str, obj: str):
    """Write one triplet into Neo4j using MERGE (deduplication automatic)."""
    tx.run("""
        MERGE (s:Entity {name: $subject})
        MERGE (t:Entity {name: $object})
        MERGE (s)-[r:RELATES {type: $predicate}]->(t)
    """, subject=subject.lower().strip(),
         predicate=predicate.lower().strip(),
         object=obj.lower().strip())


def clear_graph():
    """Wipe all nodes/edges — run before rebuilding to avoid stale data."""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("🗑️  Graph cleared")


# ── Build the graph from the first 30 chunks ──
# (30 chunks ≈ 7-10 pages; enough to build a rich graph for the demo)
N_CHUNKS   = 30
demo_chunks = chunks[:N_CHUNKS]

clear_graph()

print(f"🏗️  Building knowledge graph from {N_CHUNKS} chunks...")
print(f"   Each chunk → LLM extracts triplets → stored in Neo4j\n")

total_triplets = 0
for i, chunk in enumerate(demo_chunks):
    triplets = extract_triplets(chunk.page_content)
    if triplets:
        with driver.session(database=NEO4J_DATABASE) as session:
            for t in triplets:
                if all(k in t for k in ("subject", "predicate", "object")):
                    session.execute_write(store_triplet, t["subject"], t["predicate"], t["object"])
                    total_triplets += 1
    print(f"   Chunk {i+1:02d}/{N_CHUNKS} → {len(triplets):2d} triplets extracted")

print(f"\n✅ Knowledge graph built!")
print(f"   Total triplets stored : {total_triplets}")

🗑️  Graph cleared
🏗️  Building knowledge graph from 30 chunks...
   Each chunk → LLM extracts triplets → stored in Neo4j

   Chunk 01/30 →  7 triplets extracted
   Chunk 02/30 →  6 triplets extracted
   Chunk 03/30 →  5 triplets extracted
   Chunk 04/30 →  6 triplets extracted
   Chunk 05/30 →  6 triplets extracted
   Chunk 06/30 →  5 triplets extracted
   Chunk 07/30 →  7 triplets extracted
   Chunk 08/30 →  5 triplets extracted
   Chunk 09/30 →  6 triplets extracted
   Chunk 10/30 →  7 triplets extracted
   Chunk 11/30 →  6 triplets extracted
   Chunk 12/30 →  6 triplets extracted
   Chunk 13/30 →  7 triplets extracted
   Chunk 14/30 →  5 triplets extracted
   Chunk 15/30 →  6 triplets extracted
   Chunk 16/30 →  6 triplets extracted
   Chunk 17/30 →  5 triplets extracted
   Chunk 18/30 →  5 triplets extracted
   Chunk 19/30 →  5 triplets extracted
   Chunk 20/30 →  7 triplets extracted
   Chunk 21/30 →  6 triplets extracted
   Chunk 22/30 →  6 triplets extracted
   Chunk 23/30 →  6 

In [12]:
# ── Cell 8: Inspect the Knowledge Graph ──
# Show the team what's actually stored in Neo4j

with driver.session(database=NEO4J_DATABASE) as session:

    # Count nodes and edges
    node_count = session.run("MATCH (n:Entity) RETURN count(n) AS cnt").single()["cnt"]
    edge_count = session.run("MATCH ()-[r:RELATES]->() RETURN count(r) AS cnt").single()["cnt"]

    # Sample relationships
    sample = session.run("""
        MATCH (s:Entity)-[r:RELATES]->(t:Entity)
        RETURN s.name AS source, r.type AS relation, t.name AS target
        ORDER BY s.name
        LIMIT 25
    """)
    rows = sample.data()

print("=" * 60)
print("🕸️  NEO4J KNOWLEDGE GRAPH — SNAPSHOT")
print("=" * 60)
print(f"   Nodes (entities)     : {node_count}")
print(f"   Edges (relationships): {edge_count}")
print(f"\n{'Source':<25} {'Relation':<20} {'Target'}")
print("-" * 60)
for row in rows:
    print(f"   {row['source']:<23} {row['relation']:<20} {row['target']}")

print(f"\n✅ This is the structured knowledge graph that GraphRAG will traverse")

🕸️  NEO4J KNOWLEDGE GRAPH — SNAPSHOT
   Nodes (entities)     : 25
   Edges (relationships): 49

Source                    Relation             Target
------------------------------------------------------------
   agent                   uses                 memory
   agent                   relies on            memory
   agent                   requires             memory
   agent                   requires             tools
   agent                   relies on            tools
   agent                   uses                 tools
   agent                   improves             planning
   agent                   uses                 planning
   agent                   enables              planning
   agent                   requires             planning
   agent                   relies on            planning
   agent                   enables              reasoning
   agent                   improves             reasoning
   agent                   relies on            reasoning
   

In [13]:
# ── Cell 9: GraphRAG Query Engine ──

GRAPHRAG_ANSWER_PROMPT = """You are an AI expert on LLM agents.

Use ONLY the knowledge graph relationships below to answer the question.
Each line shows: entity → relationship → entity (a fact extracted from a research paper).

Knowledge Graph Context:
{graph_context}

Question: {question}

Instructions:
- Connect the relevant entities and relationships to form a complete answer
- Explicitly mention the relationship chain when relevant
- If the graph shows A→B→C, explain that connection in your answer
- Do NOT add information not present in the graph context

Answer:"""


def graphrag_query(question: str) -> dict:
    """
    GraphRAG pipeline:
    1. Extract query entities with LLM
    2. Traverse Neo4j graph from those entities
    3. Feed graph paths to LLM for structured answer
    """

    # ── Step 1: Extract entities from query ──
    entity_prompt = f"""Extract key entities from this question about LLM agents.
Return ONLY a JSON array of lowercase entity names: ["entity1", "entity2"]
Focus on: agent, memory, tools, planning, reasoning, task, execution, perception
Question: {question}
JSON:"""
    entity_response = extract_llm.invoke(entity_prompt)
    entity_raw      = entity_response.content.strip()
    entity_match    = re.search(r'\[.*?\]', entity_raw, re.DOTALL)
    entities        = json.loads(entity_match.group()) if entity_match else []

    # ── Step 2: Traverse graph ──
    graph_paths = []
    with driver.session(database=NEO4J_DATABASE) as session:

        # Direct 1-hop paths from/to matching entities
        result = session.run("""
            MATCH (s:Entity)-[r:RELATES]->(t:Entity)
            WHERE ANY(e IN $entities
                  WHERE toLower(s.name) CONTAINS e
                     OR toLower(t.name) CONTAINS e)
            RETURN s.name AS source, r.type AS relation, t.name AS target
            LIMIT 25
        """, entities=entities)
        for rec in result:
            graph_paths.append(f"{rec['source']} → {rec['relation']} → {rec['target']}")

        # 2-hop: find entities connected to the matched entities
        if entities:
            hop2 = session.run("""
                MATCH (s:Entity)-[r1:RELATES]->(mid:Entity)-[r2:RELATES]->(t:Entity)
                WHERE ANY(e IN $entities WHERE toLower(s.name) CONTAINS e)
                RETURN s.name AS s, r1.type AS r1, mid.name AS mid,
                       r2.type AS r2, t.name AS t
                LIMIT 15
            """, entities=entities)
            for rec in hop2:
                path = (f"{rec['s']} → {rec['r1']} → {rec['mid']} "
                        f"→ {rec['r2']} → {rec['t']}")
                if path not in graph_paths:
                    graph_paths.append(path)

    # ── Step 3: Generate answer from graph context ──
    if not graph_paths:
        graph_context = "No relevant graph relationships found for these entities."
    else:
        graph_context = "\n".join([f"  • {p}" for p in graph_paths])

    answer_prompt = GRAPHRAG_ANSWER_PROMPT.format(
        graph_context=graph_context,
        question=question
    )
    answer = extract_llm.invoke(answer_prompt)

    return {
        "answer":       answer.content,
        "entities":     entities,
        "graph_paths":  graph_paths
    }

print("✅ GraphRAG query engine ready")
print("   Pipeline: query → entity extraction → Neo4j traversal → LLM answer")

✅ GraphRAG query engine ready
   Pipeline: query → entity extraction → Neo4j traversal → LLM answer


---
## 🔹 PART 5: GRAPHRAG QUERIES

### 🟢 Simple Queries — GraphRAG handles these too

GraphRAG works on direct queries, but for straightforward factual questions it retrieves graph paths that aren't strictly necessary. The answers are comparable to RAG but the system does more work. This is the "overkill" scenario — useful to see so we know when *not* to reach for GraphRAG.

In [14]:
# ── Cell 10: GraphRAG — Simple Queries ──

print("=" * 65)
print("🟢  GraphRAG — SIMPLE QUERIES")
print("=" * 65)

for q in simple_queries:
    result = graphrag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"\n   Entities extracted: {result['entities']}")
    print(f"\n   Graph paths retrieved ({len(result['graph_paths'])}):")
    for path in result["graph_paths"][:5]:
        print(f"     • {path}")
    print(f"\n💬 GraphRAG Answer:\n{result['answer']}")
    print("\n" + "-" * 65)

🟢  GraphRAG — SIMPLE QUERIES

❓ Query: What is an autonomous agent?

   Entities extracted: ['agent']

   Graph paths retrieved (40):
     • agent → uses → memory
     • agent → relies on → memory
     • agent → requires → memory
     • agent → requires → tools
     • agent → relies on → tools

💬 GraphRAG Answer:
An autonomous agent is an entity that perceives its environment and interacts with it to execute actions. The agent requires memory to store tasks, experiences, and reflections, which enables its planning capabilities. Specifically, the agent uses memory, which stores tasks, experiences, and reflections, to enable planning. This relationship can be expressed as:

- agent → uses → memory → enables → planning

Furthermore, the agent relies on memory to support its planning process, as indicated by the relationship:

- agent → relies on → memory → enables → planning

The agent also requires planning to improve its reasoning and to enable action and execution. This is shown throug

---
### 🔴 Multi-hop Queries — GraphRAG Answers Correctly

These are the same queries RAG failed on. GraphRAG's output will:
- Show the actual entity paths retrieved from Neo4j
- Use those paths as structured context — not inferred connections
- Answer *how* and *why* by following the relationship chain explicitly

Compare the depth of these answers with what RAG produced.

In [15]:
# ── Cell 11: GraphRAG — Multi-hop Queries (shows the ADVANTAGE) ──

print("=" * 65)
print("🟢  GraphRAG — MULTI-HOP QUERIES  (watch GraphRAG reason over connections)")
print("=" * 65)

for q in multihop_queries:
    result = graphrag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"\n   Entities extracted : {result['entities']}")
    print(f"\n   Graph paths found ({len(result['graph_paths'])}):")
    for path in result["graph_paths"]:
        print(f"     • {path}")
    print(f"\n💬 GraphRAG Answer:\n{result['answer']}")
    print("\n" + "-" * 65)

print("""
✅ GraphRAG ADVANTAGE OBSERVED:
   ✓ Explicitly shows the relationship chain (path)
   ✓ Answers the "how" by following edges: A → rel → B → rel → C
   ✓ LLM reasons over structured facts, not guessed connections
   ✓ Each answer is traceable back to specific extracted relationships
""")

🟢  GraphRAG — MULTI-HOP QUERIES  (watch GraphRAG reason over connections)

❓ Query: How does memory help planning?

   Entities extracted : ['memory', 'planning']

   Graph paths found (21):
     • agent → uses → memory
     • agent → relies on → memory
     • agent → requires → memory
     • agent → improves → planning
     • agent → uses → planning
     • agent → enables → planning
     • agent → requires → planning
     • agent → relies on → planning
     • memory → enables → planning
     • memory → stores → tasks
     • memory → stores → experiences
     • memory → stores → reflections
     • planning → requires → reasoning
     • planning → enables → action
     • planning → enables → execution
     • planning → improves → execution
     • memory → enables → planning → requires → reasoning
     • memory → enables → planning → enables → action
     • memory → enables → planning → enables → execution
     • memory → enables → planning → improves → execution
     • planning → requir

---
## 🔹 PART 5.5: HYBRID RAG PIPELINE

### What is Hybrid RAG?

Hybrid RAG combines **both** retrieval strategies in a single pipeline:

```
User Question
      │
      ├──► 🔵 Vector Search (ChromaDB)  ──► Top-k semantically similar chunks
      │
      └──► 🟢 Graph Traversal (Neo4j)   ──► Entity relationship paths
                         │
                         ▼
               Merge both contexts
                         │
                         ▼
                LLM generates answer
             grounded in BOTH similarity
             AND structural relationships
```

**Why use it:**
- Vector retrieval surfaces passages that *feel like* the question (even if entities aren't explicitly named).
- Graph traversal surfaces explicit multi-hop chains between entities.
- Together they give the LLM maximum coverage for complex queries.

> ⚠️ **Trade-off:** Hybrid RAG is the most capable approach, but also the most expensive — it doubles infra costs (vector DB + graph DB) and has higher query latency. Use it when answer quality is worth the cost.

**See `CONCEPTS.md` for detailed trade-off analysis.**


In [ ]:
# ── Cell 11.0: Build the Hybrid RAG Query Engine ──
# Hybrid RAG = Vector retrieval (Chroma) + Graph traversal (Neo4j) combined.
# The LLM receives BOTH: semantically similar chunks AND graph entity paths.

HYBRID_ANSWER_PROMPT = """You are an AI expert on LLM agents.

Use ONLY evidence below to answer the question. Do not add external knowledge.

── VECTOR EVIDENCE (semantically similar text chunks) ──
{vector_context}

── GRAPH EVIDENCE (entity relationship paths from knowledge graph) ──
{graph_context}

Question: {question}

Synthesise a precise answer using both evidence sources above."""

def hybrid_rag_query(question: str) -> dict:
    """Run vector retrieval AND graph traversal, merge both contexts, generate answer."""

    # ── 1. Vector retrieval (same as RAG) ──
    docs = retriever.get_relevant_documents(question)
    vector_context = "\n\n".join([d.page_content for d in docs])

    # ── 2. Graph retrieval (same entity extraction as GraphRAG) ──
    entity_prompt = f"Extract the main named entities from this question. Return a JSON list of strings only.\nQuestion: {question}"
    entity_response = extract_llm.invoke(entity_prompt)
    try:
        raw = entity_response.content.strip()
        entities = json.loads(raw[raw.find("["):raw.rfind("]") + 1])
    except Exception:
        entities = [question.split()[0]]

    graph_paths = []
    with driver.session(database=NEO4J_DATABASE) as session:
        for entity in entities[:3]:
            # 1-hop
            result = session.run(
                """MATCH (n:Entity)-[r]->(m:Entity)
                   WHERE toLower(n.name) CONTAINS toLower($name)
                      OR toLower(m.name) CONTAINS toLower($name)
                   RETURN n.name AS src, type(r) AS rel, m.name AS tgt
                   LIMIT 10""",
                name=entity
            )
            for row in result:
                graph_paths.append(f"{row['src']} --[{row['rel']}]--> {row['tgt']}")

            # 2-hop
            result = session.run(
                """MATCH (a:Entity)-[r1]->(b:Entity)-[r2]->(c:Entity)
                   WHERE toLower(a.name) CONTAINS toLower($name)
                   RETURN a.name AS a, type(r1) AS r1, b.name AS b, type(r2) AS r2, c.name AS c
                   LIMIT 8""",
                name=entity
            )
            for row in result:
                graph_paths.append(
                    f"{row['a']} --[{row['r1']}]--> {row['b']} --[{row['r2']}]--> {row['c']}"
                )

    graph_context = "\n".join(graph_paths) if graph_paths else "No graph paths found."

    # ── 3. Generate answer using both contexts ──
    prompt = ChatPromptTemplate.from_template(HYBRID_ANSWER_PROMPT)
    chain  = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "vector_context": vector_context,
        "graph_context":  graph_context,
        "question":       question
    })

    return {"answer": answer, "chunks": docs, "graph_paths": graph_paths}

print("✅ hybrid_rag_query() ready.")
print("   → Combines Chroma vector search + Neo4j graph traversal in a single call.")


---
### 🟢 Simple Queries — Hybrid RAG handles these too

For direct factual questions Hybrid RAG also scores well, though with higher latency and cost than pure RAG since it runs both the vector lookup and the graph traversal.


In [ ]:
# ── Cell 11a: Hybrid RAG — Simple Queries ──

print("=" * 65)
print("🟣  Hybrid RAG — SIMPLE QUERIES")
print("=" * 65)

for q in simple_queries:
    result = hybrid_rag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"──────────────────────────────────────────────────────────")
    print(result["answer"])
    print(f"\n  📎 Vector chunks: {len(result['chunks'])}  |  🕸️  Graph paths: {len(result['graph_paths'])}")


---
### 🔴 Multi-hop Queries — Hybrid RAG has Maximum Coverage

The same queries RAG failed on and GraphRAG improved. Hybrid RAG runs both retrievals and the LLM sees both the semantically similar chunks **and** the graph paths.


In [ ]:
# ── Cell 11b: Hybrid RAG — Multi-hop Queries ──

print("=" * 65)
print("🟣  Hybrid RAG — MULTI-HOP QUERIES  (vector + graph combined)")
print("=" * 65)

for q in multihop_queries:
    result = hybrid_rag_query(q)
    print(f"\n❓ Query: {q}")
    print(f"──────────────────────────────────────────────────────────")
    print(result["answer"])
    print(f"\n  📎 Vector chunks used  : {len(result['chunks'])}")
    print(f"  🕸️  Graph paths used    : {len(result['graph_paths'])}")
    if result["graph_paths"]:
        for p in result["graph_paths"][:5]:
            print(f"       • {p}")


---
## 🔹 PART 6: THREE-WAY SIDE-BY-SIDE COMPARISON

🔵 **RAG** vs 🟢 **GraphRAG** vs 🟣 **Hybrid RAG** — all three run on the same multi-hop queries. This directly shows how retrieval strategy affects answer quality and the type of evidence each system used.


In [ ]:
# ── Cell 12: Side-by-Side Comparison — RAG vs GraphRAG vs Hybrid RAG ──

def compare(question: str):
    """Run RAG, GraphRAG, and Hybrid RAG on the same query and display side-by-side."""
    rag_result      = rag_query(question)
    graphrag_result = graphrag_query(question)
    hybrid_result   = hybrid_rag_query(question)

    width = 65
    print("\n" + "█" * width)
    print(f"  QUERY: {question}")
    print("█" * width)

    print(f"\n{'─'*width}")
    print("  🔵  RAG ANSWER  (vector similarity only)")
    print(f"{'─'*width}")
    print(rag_result["answer"])
    print(f"\n  📎 RAG used {len(rag_result['chunks'])} text chunks (similarity-based)")

    print(f"\n{'─'*width}")
    print("  🟢  GraphRAG ANSWER  (graph traversal only)")
    print(f"{'─'*width}")
    print(graphrag_result["answer"])
    print(f"\n  🕸️  GraphRAG used {len(graphrag_result['graph_paths'])} graph paths:")
    for path in graphrag_result["graph_paths"][:8]:
        print(f"       • {path}")

    print(f"\n{'─'*width}")
    print("  🟣  Hybrid RAG ANSWER  (vector + graph combined)")
    print(f"{'─'*width}")
    print(hybrid_result["answer"])
    print(f"\n  📎 Vector chunks: {len(hybrid_result['chunks'])}  |  🕸️  Graph paths: {len(hybrid_result['graph_paths'])}")
    for path in hybrid_result["graph_paths"][:5]:
        print(f"       • {path}")

    print(f"\n{'═'*width}\n")


# ── Run three-way comparison on multi-hop queries ──
comparison_queries = [
    "How does memory help planning?",
    "How are tools, memory, and planning connected?",
    "How do tools improve reasoning via planning?"
]

for q in comparison_queries:
    compare(q)



█████████████████████████████████████████████████████████████████
  QUERY: How does memory help planning?
█████████████████████████████████████████████████████████████████

─────────────────────────────────────────────────────────────────
  🔵  RAG ANSWER
─────────────────────────────────────────────────────────────────
Memory helps planning by storing reflections and plans, which are then included in the retrieval process. This allows the agent to consider observations, reflections, and plans together, enabling it to draw conclusions about itself and others to guide its behavior more effectively. Specifically, as mentioned in Chunk 1, reflections and plans are fed back into the memory stream to influence the agent’s future behavior, thereby enhancing the planning process.

  📎 RAG used 3 text chunks (similarity-based)

─────────────────────────────────────────────────────────────────
  🟢  GraphRAG ANSWER
─────────────────────────────────────────────────────────────────
Memory helps pl

---
## 🔹 PART 7: PROOF — GRAPHRAG ADVANTAGES

### 1. ⭐ Multi-hop Reasoning (Most Important)

**What it means:**
- RAG can only retrieve chunks *similar* to the query
- GraphRAG follows chains: A → B → C → D
- This enables answering questions that require connecting multiple facts

**Practical example from our demo:**
```
Query: "How do tools improve reasoning via planning?"

RAG path:  query → "tools" chunks ← retrieved ← similar text  ← fragmented

GraphRAG:  query → extract ["tools", "planning", "reasoning"]
           → Neo4j: tools → used by → agent
                    agent → uses → planning
                    planning → improves → reasoning
           → Answer: tools improve reasoning by enabling agents to use 
                     planning, which then improves reasoning quality
```

---

### 2. Structured Knowledge (Not Just Raw Text)

- Every fact is stored as a discrete (subject, predicate, object) triple
- No ambiguity: "X uses Y" is an explicit edge, not inferred from text
- The graph can be audited, corrected, and extended independent of the LLM

---

### 3. Explainability — Show Your Work

```
RAG:      "Here's the answer" ← no explanation of why chunks were chosen
GraphRAG: "Here's the answer, derived from these paths:
           agent → relies on → memory
           memory → stores → past actions
           past actions → inform → planning"
```
The graph paths are audit trails. You can trace every claim.

---

### 4. Better Context Retrieval for Connected Concepts

- If you query "memory", graph traversal also finds: "past actions", "experience", "episodic storage"
- These related concepts are returned as neighbors, even if they're not similar to the query text
- RAG would only return chunks containing the word "memory"

---
## 🔹 PART 8: PROOF — GRAPHRAG LIMITATIONS

### 1. Complex Setup

```
RAG setup:
  pip install langchain chromadb openai
  load → split → embed → index → done ✅  (~10 lines)

GraphRAG setup:
  pip install langchain neo4j openai chromadb
  load → split → extract entities per chunk (LLM call!) → store in Neo4j
  → build query engine → Cypher queries → done ✅  (~100+ lines)
```

**Real cost:** Every chunk requires 1 LLM call during ingestion. For 100-page docs, this is expensive and slow.

---

### 2. Slower Performance

| Stage | RAG | GraphRAG |
|-------|-----|----------|
| Ingestion | Embed (fast) | LLM extract per chunk (slow) |
| Query | Cosine similarity (ms) | Cypher query (50-200ms) |
| Total latency | ~500ms | ~2-5 seconds |

---

### 3. Extraction Quality Dependency

**If the LLM extracts wrong relationships → wrong graph → wrong answers**

```
Correct: agent → relies on → memory
Wrong:   memory → relies on → agent      ← reversed direction!
Wrong:   agent → uses → memory           ← wrong predicate
Worse:   the_system → has_component → memory module  ← vague/inconsistent
```

- Entity names must be normalized: "LLM" vs "large language model" vs "GPT" = same concept
- Without normalization, you get 3 disconnected nodes that should be 1
- This is a real production challenge

---

### 4. Not Needed for Simple Queries

**Counter-demo:** Re-run a simple query comparison:
- "What is memory in agents?" → both RAG and GraphRAG produce similar answers
- GraphRAG adds latency and complexity for zero extra value
- Over-engineering sign: using GraphRAG because it's cool, not because the problem needs it

---

### 5. Graph Schema Design is Hard

In production:
- You need to define what entity *types* to extract: `Person`, `Concept`, `Tool`, `Action`
- You need controlled vocabularies for relationship types
- Without this, you get a flat `Entity` graph that's hard to query efficiently
- This is a real data engineering problem that adds weeks to projects

In [17]:
# ── Cell 13: Demonstrate the Extraction Quality Issue ──
# This cell proves limitation #3 practically

print("=" * 65)
print("⚠️  GRAPHRAG LIMITATION DEMO — Extraction Quality")
print("=" * 65)

# Show a sample chunk and what the LLM extracted from it
sample_chunk_text = demo_chunks[5].page_content
triplets          = extract_triplets(sample_chunk_text)

print(f"\n📄 Sample Chunk Text (first 400 chars):")
print(f"   {sample_chunk_text[:400]}...")

print(f"\n🤖 LLM-Extracted Triplets:")
for t in triplets:
    print(f"   {t.get('subject','?'):<22} ──{t.get('predicate','?'):<18}──► {t.get('object','?')}")

print(f"""
💡 WHAT THIS SHOWS:
   ✓ LLM correctly extracts clear relationships
   ✗ Entity names may vary in casing, phrasing, specificity
   ✗ If two chunks call the same concept differently, they form separate nodes
   ✗ This is why production GraphRAG needs entity normalization pipelines

→ "large language model" ≠ "LLM" ≠ "GPT-4" in the graph (3 separate nodes!)
→ This is the biggest production challenge with GraphRAG
""")

⚠️  GRAPHRAG LIMITATION DEMO — Extraction Quality

📄 Sample Chunk Text (first 400 chars):
   autonomously spread invitations to the party over the next two
arXiv:2304.03442v2  [cs.HC]  6 Aug 2023...

🤖 LLM-Extracted Triplets:
   agent                  ──uses              ──► memory
   agent                  ──relies on         ──► tools
   agent                  ──requires          ──► planning
   agent                  ──improves          ──► reasoning
   agent                  ──executes          ──► action

💡 WHAT THIS SHOWS:
   ✓ LLM correctly extracts clear relationships
   ✗ Entity names may vary in casing, phrasing, specificity
   ✗ If two chunks call the same concept differently, they form separate nodes
   ✗ This is why production GraphRAG needs entity normalization pipelines

→ "large language model" ≠ "LLM" ≠ "GPT-4" in the graph (3 separate nodes!)
→ This is the biggest production challenge with GraphRAG



---
## 🔹 PART 9: RAG ADVANTAGES

RAG is the right tool for the majority of real-world use cases. This section exists because the demo can make it look like RAG is broken — it's not. It's fast, cheap, and works well for most questions.

| Advantage | Detail |
|-----------|--------|
| ✅ **Simple to implement** | ~10 lines of code vs 100+ for GraphRAG |
| ✅ **Fast** | Cosine similarity is ms-level; graph traversal adds overhead |
| ✅ **Works for factual queries** | Single-paragraph answers, direct lookups |
| ✅ **No extraction step** | No LLM calls during ingestion, no risk of incorrect triplets |
| ✅ **Mature ecosystem** | ChromaDB, Pinecone, Weaviate, FAISS — battle tested |
| ✅ **Lower cost** | No graph DB license, no extraction API calls |
| ✅ **Easier to debug** | "Why this answer?" → check which chunks were retrieved |

**RAG works best for:**
- Customer support FAQ bots
- Document search ("find me the section about X")
- Summarisation of single documents
- Search over product catalogs, HR policies, legal clauses
- Prototypes and MVPs

---

## 🔹 WHEN TO USE WHAT

```
                    ┌─────────────────────────────────────┐
                    │     What type of query is this?     │
                    └──────────────┬──────────────────────┘
                                   │
               ┌───────── "What is X?" ──────────┐
               ▼                                 ▼
        Single concept,                   Connecting multiple
        direct answer                     concepts, "why/how"
               │                                 │
               ▼                                 ▼
           Use RAG ✅                    Does the domain have
        Fast, simple,                    rich relationships?
        sufficient                               │
                                  ┌────── YES ───┴── NO ──────┐
                                  ▼                            ▼
                           Use GraphRAG ✅                Use RAG ✅
                        (relationships = answer)      (relationships not needed)
```

| Use RAG | Use GraphRAG |
|---------|-------------|
| Simple Q&A, FAQ | Multi-hop reasoning |
| Direct factual queries | Domain: medical, legal, finance |
| Speed is critical | Explainability required |
| Low budget project | Knowledge spans many documents |
| Prototype/MVP | Production AI assistant |
| One document | Enterprise knowledge base |

**Real-world decision examples:**

| Scenario | Right Choice | Why |
|----------|-------------|-----|
| Customer support FAQ | RAG | Direct answers, simple |
| Drug interaction checker | GraphRAG | Drug A → affects → pathway → impacts → Drug B |
| Document summariser | RAG | Summarise = similarity, not relationship |
| Legal precedent finder | GraphRAG | Case A → cites → Case B → includes → Law C |
| Resume keyword matcher | RAG | Match job description to resume text |
| Supply chain dependency map | GraphRAG | Supplier → provides → Component → required by → Product |
| Internal HR policy search | RAG | "What's the leave policy?" lives in one paragraph |
| Medical diagnosis assistant | GraphRAG | Symptom → indicates → condition → treated by → drug |

---
## 🔹 PART 10: THE RAG ECOSYSTEM HIERARCHY

RAG is not one thing — it's a family of systems. Here is the full landscape, from simplest to most powerful:

```
              ┌──────────────────────────────────────────┐
   Level 1    │  Vectorless RAG  (BM25)                  │  Keyword match only
              └────────────────────┬─────────────────────┘
                                   │  + semantic understanding (embeddings)
              ┌────────────────────▼─────────────────────┐
   Level 2    │  Vector RAG                              │  Meaning-based similarity
              └────────────────────┬─────────────────────┘
                                   │  + keyword recall (BM25 blended in)
              ┌────────────────────▼─────────────────────┐
   Level 3    │  Hybrid RAG  (BM25 + Vector)             │  Best recall, balanced
              └────────────────────┬─────────────────────┘
                                   │  + relationship / graph reasoning
              ┌────────────────────▼─────────────────────┐
   Level 4    │  GraphRAG                                │  Traverses knowledge graph
              └────────────────────┬─────────────────────┘
                                   │  + semantic + keyword + graph combined
              ┌────────────────────▼─────────────────────┐
   Level 5    │  Hybrid GraphRAG  ⭐  (Most Advanced)     │  All retrieval modes unified
              └──────────────────────────────────────────┘
```

### Personality of each system

| Level | System | Superpower | Failure mode |
|-------|--------|-----------|-------------|
| 1 | **BM25 (Vectorless)** | Blazing fast, exact match | Misses synonyms, no reasoning |
| 2 | **Vector RAG** | Understands meaning & paraphrasing | Cannot chain relationships |
| 3 | **Hybrid RAG** | Best recall, balanced accuracy | Still no graph reasoning |
| 4 | **GraphRAG** | Multi-hop reasoning, explainable | Slow, rare in prod, extraction-dependent |
| 5 | **Hybrid GraphRAG** | Best of everything | Most expensive, hardest to build |

---

### Key Takeaway

> *"Vectorless RAG relies purely on keywords, vector RAG adds semantic understanding, GraphRAG adds relationships, and Hybrid GraphRAG combines everything for the most robust system."*

---

### Session Closing Summary

> *"RAG is not one thing — it's a family of systems. BM25 is fast but shallow. Vector RAG understands meaning but can't reason across documents. GraphRAG adds relationship intelligence at the cost of complexity. In production, the choice is always a balance: latency vs. reasoning vs. cost. Hybrid GraphRAG is the most powerful, but the right system depends entirely on what the query actually needs."*

---
## 🔹 PART 11: PRODUCTION-GRADE COMPARISON TABLE

40 dimensions covering retrieval mechanics, accuracy, latency, cost, and production readiness across all RAG variants — including Vectorless RAG for full context.

| # | Dimension | BM25 (Vectorless) | Vector RAG | Pure GraphRAG | Hybrid RAG | Hybrid GraphRAG | Vectorless RAG |
|---|-----------|:-----------------:|:----------:|:-------------:|:----------:|:---------------:|:--------------:|
| 1 | Retrieval type | Keyword match | Semantic similarity | Graph traversal | Keyword + semantic | Semantic + graph | Keyword match |
| 2 | Uses embeddings | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ❌ No |
| 3 | Uses graph | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 4 | Data structure | Inverted index | Vector DB | Graph DB | Hybrid index | Graph + vector DB | Inverted index |
| 5 | Query handling | Exact terms | Meaning-based | Entity extraction | Both | Both | Exact terms |
| 6 | Matching style | String match | Vector similarity | Node/path traversal | Combined | Combined | String match |
| 7 | Multi-hop reasoning | ❌ No | ❌ Weak | ✅ Strong | ❌ Weak | ✅ Strong | ❌ No |
| 8 | Context understanding | Low | Medium | High | Medium | High | Low |
| 9 | Explainability | Medium | Low | High | Medium | High | Medium |
| 10 | Setup complexity | Low | Medium | High | Medium | Very High | Low |
| 11 | **Latency (query time)** | ⚡ Very low | ⚡ Low | 🐢 High | ⚡ Medium | 🐢 High | ⚡ Very low |
| 12 | Throughput | High | High | Medium | High | Medium | High |
| 13 | **Cost (infra)** | 💚 Low | 🟡 Medium | 🟡 Medium | 🟡 Medium | 🔴 High | 💚 Low |
| 14 | **Cost (compute)** | 💚 Low | 🟡 Medium | 🔴 High | 🟡 Medium | 🔴 High | 💚 Low |
| 15 | **Storage cost** | 💚 Low | 🟡 Medium | 🟡 Medium | 🟡 Medium | 🔴 High | 💚 Low |
| 16 | Accuracy — simple Qs | Medium | High | High | High | High | Medium |
| 17 | **Accuracy — complex Qs** | Low | Medium | High | Medium | Very High | Low |
| 18 | Handles paraphrasing | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ❌ No |
| 19 | Dependency on wording | High | Low | High | Medium | Low | High |
| 20 | Dependency on structure | Low | Low | High | Low | Medium | Low |
| 21 | Recall (coverage) | Low | High | Low | High | Very High | Low |
| 22 | Precision | Medium | High | High | High | High | Medium |
| 23 | **Hallucination risk** | High | Medium | Low | Medium | Low | High |
| 24 | Preprocessing needed | Low | Medium | High | Medium | High | Low |
| 25 | Entity extraction needed | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 26 | Relationship extraction | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 27 | Handles noisy data | Poor | Good | Poor | Good | Good | Poor |
| 28 | **Failure mode** | Misses synonyms | Misses relationships | Misses entities | Balanced | Rare | Misses synonyms |
| 29 | Scalability | High | High | Medium | High | Medium | High |
| 30 | Maintenance effort | Low | Medium | High | Medium | High | Low |
| 31 | Engineering effort | Low | Medium | High | Medium | Very High | Low |
| 32 | Debuggability | Medium | Low | High | Medium | High | Medium |
| 33 | Observability | Medium | Medium | High | Medium | High | Medium |
| 34 | **Production maturity** | High | Very High | Low | High | Medium | Medium |
| 35 | **Production adoption** | Common | Very common | Rare | Common | Growing | Limited |
| 36 | **Real-time suitability** | Excellent | Good | Limited | Good | Limited | Excellent |
| 37 | Batch processing | Good | Good | Excellent | Good | Excellent | Good |
| 38 | Data freshness handling | Easy | Medium | Hard | Medium | Hard | Easy |
| 39 | Incremental updates | Easy | Medium | Hard | Medium | Hard | Easy |
| 40 | **Best use case** | Search | Q&A | Knowledge graphs | Enterprise search | Advanced AI | Basic retrieval |

---

### Key Trade-off Summary

| Priority | Best choice |
|----------|-------------|
| Fastest response | BM25 |
| Best semantic understanding | Vector RAG |
| Best reasoning + explainability | GraphRAG |
| Best recall balanced with accuracy | Hybrid RAG |
| Best overall (cost not a constraint) | Hybrid GraphRAG |

---

> *"BM25 is fast but shallow, RAG is semantic, GraphRAG is relational, and Hybrid GraphRAG is the most powerful but most expensive."*

---
## 🔹 PART 12: KEY PRODUCTION INSIGHTS

Three dimensions drive almost every architecture decision in production AI systems: **latency**, **cost**, and **maturity**. Here's the full breakdown.

---

### 🔴 Dimension A: Latency (response time)

```
Fastest ──────────────────────────────────────────► Slowest

BM25          Vector RAG      Hybrid RAG      GraphRAG / Hybrid GraphRAG
 ⚡                ⚡              ⚡ med              🐢 slow
~1–5 ms         ~50–200 ms      ~100–400 ms         ~500 ms–5 s
```

**Why is GraphRAG slow?**
1. LLM call: extract entities from the query (~300ms)
2. Cypher traversal: multi-hop paths through Neo4j (~100–500ms)
3. LLM call: generate answer from graph paths (~500ms+)

**Why is BM25 extremely fast?**
- Inverted index lookup — no neural network, no API call, pure text matching
- Same tech powering Elasticsearch and traditional search engines

Graph traversal increases latency significantly compared to vector search. For real-time systems that need sub-500ms responses, GraphRAG alone is not the right choice.

---

### 🔴 Dimension B: Cost

```
Cheapest ─────────────────────────────────────────► Most expensive

BM25         Vector RAG       Hybrid RAG       GraphRAG      Hybrid GraphRAG
 💚              🟡               🟡              🔴              🔴🔴
```

**Where GraphRAG cost comes from:**

| Cost Source | Detail |
|-------------|--------|
| LLM calls during ingestion | 1 API call per chunk — expensive at scale |
| Graph DB hosting | Neo4j Aura, Amazon Neptune — not free at production scale |
| Vector DB hosting | Still needed in Hybrid GraphRAG |
| Compute (graph ops) | Graph traversal is CPU/memory intensive |
| Storage | Stores text + vectors + graph — highest of all options |
| Engineering time | Building + maintaining the extraction pipeline |

GraphRAG has hidden costs beyond the API calls. The graph DB, the extraction pipeline, and the engineering time to keep it clean all add up — especially when the graph needs to stay current.

---

### 🔴 Dimension C: Production Capability / Maturity

| System | Reality today |
|--------|--------------|
| **BM25** | Powers most search engines. Proven at Google/Elasticsearch scale. |
| **Vector RAG** | Most widely deployed — ChatGPT plugins, enterprise Q&A, copilots |
| **Hybrid RAG** | Common in enterprise search (Elastic, Azure AI Search) |
| **Pure GraphRAG** | Rare in production — mainly research & specialised domains |
| **Hybrid GraphRAG** | Emerging — Microsoft GraphRAG (2024), Neo4j GenAI pilots |

Vector-based RAG is the most widely adopted system in production today. GraphRAG is still emerging and mainly used in advanced or specialised systems.

---

### 🎤 Key Takeaway

> *"In production, systems are designed by balancing latency, cost, and reasoning capability. While GraphRAG provides strong reasoning, hybrid approaches are preferred to maintain recall and performance. Pure GraphRAG is rare in production — Hybrid GraphRAG is the direction the industry is moving toward."*

---
## 🔹 PART 13: DEEP DIVE — PRODUCTION ADOPTION & REAL-TIME SUITABILITY

Rows 35 and 36 of the comparison table often need deeper explanation. Here's what each means in practice.

---

### 📌 Point 35 — Production Adoption

**Definition:** How commonly is each system actually deployed in real production systems today?

```
Very Common ─────────────────────────────────────► Rare

Vector RAG     Hybrid RAG     BM25       Hybrid GraphRAG   Pure GraphRAG
Very common     Common       Common        Growing              Rare
```

#### 🟢 Very Common — Vector RAG & Hybrid RAG

Used in:
- Enterprise chatbots (Microsoft Copilot, GitHub Copilot)
- Customer service AI (Zendesk, Intercom with AI)
- Document Q&A products (Notion AI, Confluence AI)

**Why this is the most deployed:**
- Good enough accuracy for 80% of use cases
- Cost and latency are acceptable
- Mature libraries: LangChain, LlamaIndex, Azure AI Search

#### 🟡 Growing — Hybrid GraphRAG

Used in:
- Microsoft's GraphRAG open-source project (2024)
- Neo4j enterprise AI integrations
- Pharmaceutical companies (drug–disease–gene knowledge graphs)
- Large financial institutions (regulatory knowledge bases)

**Why it's growing but not everywhere yet:**
- Complex setup — needs a dedicated data engineering effort
- High cost — most companies start with Vector RAG and add graph reasoning only when they hit its limits

#### 🔴 Rare — Pure GraphRAG

- Requires a perfect, well-maintained graph
- Without fallback vector retrieval, recall drops for anything not in the graph
- Most teams that "do GraphRAG" are actually doing Hybrid GraphRAG

#### 🔴 Limited — BM25 / Vectorless for LLM-based systems

- BM25 is alive in pure text search (not LLM-based Q&A)
- Fails on paraphrasing and misses semantic context
- Mostly used as a baseline benchmark or legacy component

---

### 📌 Point 36 — Real-time Suitability

**Definition:** Can the system respond fast enough for live, interactive use cases (chatbots, copilots, voice interfaces)?

#### ⚡ Excellent — BM25, Vectorless RAG

- Simple keyword lookup: 1–5 milliseconds
- No neural network, no API call
- Handles thousands of queries per second
- Works for: autocomplete, search-as-you-type, instant results

#### ⚡ Good — Vector RAG, Hybrid RAG

- Embedding + cosine similarity: ~50–200 ms
- Optimised hardware (FAISS, ANN search) makes it near-real-time
- Used in: chatbots, co-pilots, AI assistants
- The small latency is imperceptible to users in most cases

#### 🐢 Limited — GraphRAG, Hybrid GraphRAG

Step-by-step breakdown of why GraphRAG is slower:

```
User query:  "How does memory help planning?"
       │
Step 1 │  LLM call:  extract entities ["memory", "planning"]     ~300ms
       │
Step 2 │  Cypher:    traverse Neo4j graph, find paths            ~100–500ms
       │
Step 3 │  LLM call:  generate answer from graph context          ~500ms–2s
       │
Total time:  ~1–3 seconds
```

Compare with Vector RAG:

```
User query → embed → cosine similarity → top-3 chunks → LLM answer
              ~20ms       ~50ms                              ~500ms
Total time: ~600ms
```

GraphRAG is less suitable for strict real-time systems. Where sub-second responses are required consistently, vector-based RAG is the practical choice.

---

### 🔥 Combined Insight

> *"There is a fundamental trade-off between reasoning depth and response speed. GraphRAG improves reasoning quality but increases latency and is still rare in production. Vector RAG hits the sweet spot — good enough reasoning, fast enough for real-time, widely supported. GraphRAG is where the field is heading, but vector RAG is where it lives today."*

---

### ⚡ One-line Summary

```
Row 35 — Adoption   = "How many real systems use it?"  → Vector RAG wins
Row 36 — Real-time  = "How fast can it respond?"       → BM25/Vector wins, GraphRAG struggles
```

In [18]:
# ── Cell 14: Final Summary — Print All Results ──
# Run this at the end to generate a clean summary for the team

print("=" * 72)
print("  📊  GRAPHRAG SESSION — FINAL SUMMARY")
print("=" * 72)

print("""
┌───────────────────────────────────────────────────────────────────────┐
│  RAG ECOSYSTEM HIERARCHY                                               │
│  Level 1: BM25 (Vectorless)   → keyword match, fastest, cheapest      │
│  Level 2: Vector RAG          → semantic understanding, widely used    │
│  Level 3: Hybrid RAG          → BM25 + vectors, best recall            │
│  Level 4: GraphRAG            → graph reasoning, explainable, slow     │
│  Level 5: Hybrid GraphRAG ⭐   → everything combined, most powerful    │
└───────────────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────────────┐
│  RAG (Retrieval-Augmented Generation)                                  │
│  ─────────────────────────────────────────────────────────────────── │
│  ✅  Simple: chunk → embed → similarity → answer                       │
│  ✅  Fast, cheap, widely supported                                      │
│  ✅  Great for direct factual queries                                   │
│  ❌  Cannot reason over multi-hop relationships                        │
│  ❌  Cannot connect concepts from different document sections          │
│  ❌  Not explainable (why these chunks?)                               │
└───────────────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────────────┐
│  GraphRAG (Graph + Retrieval-Augmented Generation)                     │
│  ─────────────────────────────────────────────────────────────────── │
│  ✅  Extracts entities + relationships → builds knowledge graph        │
│  ✅  Traverses graph to find connections between concepts              │
│  ✅  Multi-hop reasoning: agent → uses → tools → improves → reasoning  │
│  ✅  Explainable — shows the relationship path, not just an answer     │
│  ❌  Complex setup: extraction + graph DB + Cypher queries             │
│  ❌  Slower: LLM per chunk during ingestion + graph traversal overhead│
│  ❌  Quality depends entirely on extraction accuracy                   │
│  ❌  Overkill for simple Q&A                                           │
└───────────────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────────────┐
│  PRODUCTION DIMENSIONS AT A GLANCE                                     │
│  ─────────────────────────────────┬───────────┬──────────────────── │
│  Dimension                        │  RAG      │  GraphRAG            │
│  ─────────────────────────────────┼───────────┼──────────────────── │
│  Latency (query response)         │  ⚡ Fast   │  🐢 Slow             │
│  Cost (infra + compute)           │  💚 Low   │  🔴 High             │
│  Production adoption (today)      │  Very High│  Rare/Growing        │
│  Real-time suitability            │  ✅ Good  │  ⚠️  Limited          │
│  Hallucination risk               │  Medium   │  Low                 │
│  Multi-hop reasoning              │  ❌ Weak  │  ✅ Strong           │
│  Explainability                   │  Low      │  High                │
│  Setup complexity                 │  Medium   │  Very High           │
└─────────────────────────────────────────────────────────────────────┘

KEY RULES:
  → Simple question, direct answer?               Use RAG
  → "How/Why" questions about relationships?      Use GraphRAG
  → Speed critical (real-time chatbot)?           Use Vector RAG or Hybrid RAG
  → Budget limited + prototype?                   Use Vector RAG
  → Enterprise, complex domain, explainability?  Use Hybrid GraphRAG

EMERGENCY 1-LINERS:
  "RAG finds similar text. GraphRAG finds connected knowledge."
  "BM25 is fast but shallow, RAG is semantic, GraphRAG is relational,
   Hybrid GraphRAG is the most powerful but most expensive."
""")

print("=" * 72)
print("  Session demo complete ✅")
print("=" * 72)

  📊  GRAPHRAG SESSION — FINAL SUMMARY

┌────────────────────────────────────────────────────────────────────┐
│  RAG (Retrieval-Augmented Generation)                              │
│  ─────────────────────────────────────────────────────────────── │
│  ✅ Simple: chunk → embed → similarity → answer                    │
│  ✅ Fast, cheap, widely supported                                  │
│  ✅ Great for direct factual queries                               │
│  ❌ Cannot reason over multi-hop relationships                     │
│  ❌ Cannot connect concepts from different document sections       │
│  ❌ Not explainable (why these chunks?)                            │
└────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────┐
│  GraphRAG (Graph + Retrieval-Augmented Generation)                 │
│  ─────────────────────────────────────────────────────────────── │
│  ✅ Extracts entities + relationships → builds

---
## 🎤 Session Closing

---

### What We Proved Today

| Point | Evidence |
|-------|---------|
| RAG works well for direct queries | ✅ Cell 5 — "What is an autonomous agent?" answered correctly |
| RAG fails on multi-hop queries | ❌ Cell 6 — fragmented, disconnected answers |
| GraphRAG builds a knowledge graph | ✅ Cell 8 — Neo4j nodes + edges visible |
| GraphRAG answers multi-hop queries | ✅ Cell 11 — connected, path-traced answers |
| GraphRAG has a real complexity cost | ⚠️ Cell 7 — slow ingestion; Cell 13 — extraction quality issues |
| Both have valid use cases | ✅ Covered in Parts 9–13 |
| RAG is a spectrum, not one system | ✅ Part 10 — BM25 → Vector → Hybrid → GraphRAG → Hybrid GraphRAG |

---

### Team Takeaways

1. **Don't reach for GraphRAG by default** — it adds real complexity and cost
2. **GraphRAG works best when relationships are the answer** — multi-hop, cross-section reasoning
3. **Extraction quality is the biggest production risk** — invest in entity normalisation
4. **Vector RAG is the production standard today** — fast, semantic, widely supported
5. **GraphRAG is the future direction** — Hybrid GraphRAG is where the industry is heading
6. **Latency and cost drive architecture decisions** — graph traversal is slower and more expensive, but provides stronger reasoning where it matters

---

### Common Questions

| Question | Answer |
|---------|--------|
| When should we use GraphRAG? | When the question requires connecting *how/why* things relate, not just *what* something is |
| Why isn't GraphRAG used everywhere? | Complex setup, slower response, still maturing — Vector RAG covers 80% of use cases |
| What's the latency trade-off? | BM25 ~5ms, Vector RAG ~200ms, GraphRAG ~1–3s — graph traversal is the bottleneck |
| What's the best system overall? | Hybrid GraphRAG — but only if cost and complexity are justified for the use case |
| Will RAG be replaced by GraphRAG? | No — most production systems will use Hybrid approaches, not pure GraphRAG |
| What's the cheapest production option? | BM25 for search, Vector RAG for Q&A — both are mature and low cost |

---

> *"RAG finds similar text. GraphRAG finds connected knowledge."*  
> *"Vectorless RAG is keywords. Vector RAG is meaning. GraphRAG is relationships. Hybrid GraphRAG is everything."*

---
*Built with: LangChain · Neo4j Aura · ChromaDB · OpenAI GPT-4o-mini · uv*

---
## 📋 REFERENCE: Complete 40-Dimension Comparison

Full side-by-side across all six RAG variants. Use this as a reference when evaluating which system fits a given use case.

| Dimension | BM25 (Keyword RAG) | Vector RAG (Traditional) | Pure GraphRAG | Hybrid RAG (BM25 + Vector) | Hybrid GraphRAG | Vectorless RAG |
|-----------|-------------------|--------------------------|---------------|---------------------------|-----------------|----------------|
| 1. Retrieval Type | Keyword match | Semantic similarity | Graph traversal | Keyword + semantic | Semantic + graph | Keyword match |
| 2. Uses embeddings | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ❌ No |
| 3. Uses graph | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 4. Data structure | Inverted index | Vector DB | Graph DB | Hybrid index | Graph + vector | Inverted index |
| 5. Query handling | Exact terms | Meaning-based | Entity extraction | Both | Both | Exact terms |
| 6. Matching style | String match | Vector similarity | Node/path traversal | Combined | Combined | String match |
| 7. Multi-hop reasoning | ❌ No | ❌ Weak | ✅ Strong | ❌ Weak | ✅ Strong | ❌ No |
| 8. Context understanding | Low | Medium | High | Medium | High | Low |
| 9. Explainability | Medium | Low | High | Medium | High | Medium |
| 10. Setup complexity | Low | Medium | High | Medium | Very High | Low |
| 11. Latency (query time) | ⚡ Very low | ⚡ Low | 🐢 High | ⚡ Medium | 🐢 High | ⚡ Very low |
| 12. Throughput | High | High | Medium | High | Medium | High |
| 13. Cost (infra) | Low | Medium | Medium | Medium | High | Low |
| 14. Cost (compute) | Low | Medium | High (graph ops) | Medium | High | Low |
| 15. Storage cost | Low | Medium (vectors) | Medium (graph) | Medium | High (both) | Low |
| 16. Accuracy (simple Qs) | Medium | High | High | High | High | Medium |
| 17. Accuracy (complex Qs) | Low | Medium | High | Medium | Very High | Low |
| 18. Handles paraphrasing | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ✅ Yes | ❌ No |
| 19. Dependency on wording | High | Low | High | Medium | Low | High |
| 20. Dependency on structure | Low | Low | High | Low | Medium | Low |
| 21. Recall (coverage) | Low | High | Low | High | Very High | Low |
| 22. Precision | Medium | High | High | High | High | Medium |
| 23. Hallucination risk | High | Medium | Low | Medium | Low | High |
| 24. Preprocessing needed | Low | Medium | High | Medium | High | Low |
| 25. Entity extraction needed | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 26. Relationship extraction | ❌ No | ❌ No | ✅ Yes | ❌ No | ✅ Yes | ❌ No |
| 27. Handles noisy data | Poor | Good | Poor | Good | Good | Poor |
| 28. Failure mode | Misses synonyms | Misses relationships | Misses entities | Balanced | Rare | Misses synonyms |
| 29. Scalability | High | High | Medium | High | Medium | High |
| 30. Maintenance effort | Low | Medium | High | Medium | High | Low |
| 31. Engineering effort | Low | Medium | High | Medium | Very High | Low |
| 32. Debuggability | Medium | Low | High | Medium | High | Medium |
| 33. Observability | Medium | Medium | High | Medium | High | Medium |
| 34. Production maturity | High (search engines) | Very High | Low | High | Medium | Medium |
| 35. Production adoption | Common | Very common | Rare | Common | Growing | Limited |
| 36. Real-time suitability | Excellent | Good | Limited | Good | Limited | Excellent |
| 37. Batch processing | Good | Good | Excellent | Good | Excellent | Good |
| 38. Data freshness handling | Easy | Medium | Hard | Medium | Hard | Easy |
| 39. Incremental updates | Easy | Medium | Hard | Medium | Hard | Easy |
| 40. Best use case | Search | Q&A | Knowledge graphs | Enterprise search | Advanced AI | Basic retrieval |

---

### Quick Decision Rules from this Table

| Row to check | Decision it drives |
|--------------|-------------------|
| Row 7 — Multi-hop | If queries need chained reasoning → GraphRAG or Hybrid GraphRAG |
| Row 11 — Latency | If real-time response < 500ms is required → BM25 or Vector RAG |
| Row 13–15 — Cost | If budget is constrained → BM25 or pure Vector RAG |
| Row 17 — Complex Qs | If complex domain queries are the core use case → GraphRAG+ |
| Row 23 — Hallucination | If factual accuracy is critical → GraphRAG (lowest risk) |
| Row 35 — Adoption | If proven production track record matters → Vector RAG or Hybrid RAG |
| Row 36 — Real-time | If it's a chatbot or co-pilot → avoid pure GraphRAG |

---
*End of session reference.*